# UTS IN0417 – Kecerdasan Buatan dan ML Dasar
## Mini Project Individual – Paket B
**Nama:** Tyas Prio Laksono  
**NIM:** 261636004  
**Dataset:** Vehicle Dataset from CarDekho (`Car details v3.csv`)  
**Target:** `selling_price`  
**Algoritma:** Linear Regression  
**Random state:** 4

Notebook ini membandingkan model dasar (`year`, `km_driven`) dengan model yang menambahkan fitur kategorikal (`fuel`, `seller_type`, `transmission`, `owner`).

## 1. Import library dan load data
Unduh dataset dari Kaggle sesuai tautan pada soal, lalu unggah file CSV pada saat diminta.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

uploaded = files.upload()
file_name = next(iter(uploaded))
df = pd.read_csv(file_name)
print('Dataset berhasil dimuat:', file_name)

## 2. Data understanding

In [ ]:
print('Ukuran dataset:', df.shape)
print('\nNama kolom:', df.columns.tolist())
print('\nTipe data:')
print(df.dtypes)
print('\nMissing value:')
print(df.isnull().sum())
display(df.head())

## 3. Preprocessing dan pemilihan fitur
Target yang diprediksi adalah `selling_price`. Baris dengan target atau fitur dasar yang kosong dibuang karena ketiganya diperlukan. Duplikat dihapus. Missing value pada fitur numerik dan kategorikal lainnya ditangani dengan imputasi median dan modus di dalam pipeline.

In [ ]:
required = ['year', 'km_driven', 'selling_price', 'fuel', 'seller_type', 'transmission', 'owner']
missing_columns = [c for c in required if c not in df.columns]
if missing_columns:
    raise ValueError(f'Kolom tidak ditemukan: {missing_columns}')

df['selling_price'] = pd.to_numeric(df['selling_price'], errors='coerce')
df['year'] = pd.to_numeric(df['year'], errors='coerce')
df['km_driven'] = pd.to_numeric(df['km_driven'], errors='coerce')
df = df.dropna(subset=['selling_price', 'year', 'km_driven']).copy()
jumlah_awal = len(df)
df = df.drop_duplicates().copy()
print('Jumlah data setelah preprocessing:', df.shape)
print('Duplikat yang dihapus:', jumlah_awal - len(df))
fitur_dasar = ['year', 'km_driven']
fitur_kategorikal = ['fuel', 'seller_type', 'transmission', 'owner']
fitur_lengkap = fitur_dasar + fitur_kategorikal
X = df[fitur_lengkap]
y = df['selling_price']
print('Fitur dasar:', fitur_dasar)
print('Fitur tambahan:', fitur_kategorikal)
print('Target:', y.name)

## 4. Train-test split
Kedua model memakai pembagian data yang sama: 80% data latih dan 20% data uji, dengan `random_state=4` sesuai tiga digit terakhir NIM.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=4
)
print('Data train:', len(X_train))
print('Data test:', len(X_test))

## 5. Modeling
Model 1 menggunakan `year` dan `km_driven`. Model 2 menambahkan empat fitur kategorikal. Fitur kategorikal diubah menjadi numerik menggunakan One-Hot Encoding.

In [ ]:
preprocess_dasar = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), fitur_dasar)
])
model_dasar = Pipeline([
    ('preprocessing', preprocess_dasar),
    ('regression', LinearRegression())
])
model_dasar.fit(X_train[fitur_dasar], y_train)
pred_dasar = model_dasar.predict(X_test[fitur_dasar])

preprocess_lengkap = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), fitur_dasar),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]), fitur_kategorikal)
])
model_lengkap = Pipeline([
    ('preprocessing', preprocess_lengkap),
    ('regression', LinearRegression())
])
model_lengkap.fit(X_train[fitur_lengkap], y_train)
pred_lengkap = model_lengkap.predict(X_test[fitur_lengkap])
print('Kedua model selesai dilatih.')

## 6. Evaluasi model
MAE mengukur rata-rata selisih absolut antara harga aktual dan prediksi. Semakin kecil MAE semakin baik. R² menunjukkan proporsi variasi target yang dapat dijelaskan model; semakin mendekati 1 umumnya semakin baik.

In [ ]:
hasil = pd.DataFrame({
    'Model': ['Model Dasar (year, km_driven)', 'Model Lengkap (+ fitur kategorikal)'],
    'MAE': [mean_absolute_error(y_test, pred_dasar), mean_absolute_error(y_test, pred_lengkap)],
    'R2': [r2_score(y_test, pred_dasar), r2_score(y_test, pred_lengkap)]
})
display(hasil.round(3))

perbandingan = pd.DataFrame({
    'Harga Aktual': y_test.values,
    'Prediksi Model Dasar': pred_dasar,
    'Prediksi Model Lengkap': pred_lengkap
})
display(perbandingan.head(10).round(0))

## 7. Visualisasi aktual vs prediksi

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, pred_lengkap, alpha=0.45)
batas_min = min(y_test.min(), pred_lengkap.min())
batas_max = max(y_test.max(), pred_lengkap.max())
plt.plot([batas_min, batas_max], [batas_min, batas_max], 'r--')
plt.xlabel('Harga Aktual (selling_price)')
plt.ylabel('Harga Prediksi')
plt.title('Harga Aktual vs Prediksi - Model Lengkap')
plt.tight_layout()
plt.show()

## 8. Interpretasi hasil
Bandingkan nilai MAE dan R² dari tabel evaluasi. Model yang lebih baik umumnya memiliki MAE lebih rendah dan R² lebih tinggi. Hasil akhir harus dijelaskan berdasarkan angka yang muncul setelah notebook dijalankan, bukan berdasarkan perkiraan.

In [ ]:
mae_1 = mean_absolute_error(y_test, pred_dasar)
mae_2 = mean_absolute_error(y_test, pred_lengkap)
r2_1 = r2_score(y_test, pred_dasar)
r2_2 = r2_score(y_test, pred_lengkap)
print(f'Model dasar: MAE={mae_1:,.2f}; R2={r2_1:.4f}')
print(f'Model lengkap: MAE={mae_2:,.2f}; R2={r2_2:.4f}')
if mae_2 < mae_1:
    print('Model lengkap memiliki MAE lebih rendah.')
else:
    print('Model dasar memiliki MAE lebih rendah atau sama.')
if r2_2 > r2_1:
    print('Model lengkap memiliki R2 lebih tinggi.')
else:
    print('Model dasar memiliki R2 lebih tinggi atau sama.')

## 9. Jawaban konseptual dan kesimpulan
**Mengapa confusion matrix tidak tepat untuk target kontinu?** Karena `selling_price` berupa angka kontinu, bukan label kelas. Confusion matrix digunakan untuk membandingkan label aktual dan label prediksi pada masalah klasifikasi.

**Jika target dipaksa menjadi kategori, apa yang dilakukan dan informasi apa yang hilang?** Tentukan batas kategori terlebih dahulu (misalnya harga rendah, sedang, dan tinggi), lalu ubah harga menjadi label kategori. Proses ini menghilangkan detail nilai harga asli dan perbedaan antarharga yang berada dalam kategori yang sama.

**Kesimpulan:** Isi kesimpulan berdasarkan perbandingan MAE dan R² hasil aktual. Jelaskan apakah penambahan fitur kategorikal membantu model atau tidak, serta sebutkan keterbatasan bahwa prediksi hanya menggunakan fitur yang tersedia pada dataset.